In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.feature_selection import RFE
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer, PolynomialFeatures, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import os
import fiona
from geopy.distance import geodesic

In [ ]:
doubs_data = gpd.read_file("../datasets_par_departement/departement-25-doubs-original.geojson")
caserne = gpd.read_file("../data/caserne/2024_06_21_CIS.shp")
compagnies = gpd.read_file("../data/caserne/2024_06_21_Compagnies.shp")
firepoint_1 = gpd.read_file("../data/hexagones_firepoint_1.geojson")
firepoint_2 = gpd.read_file("../data/hexagones_firepoint_2.geojson")
firepoint_3 = gpd.read_file("../data/hexagones_firepoint_3.geojson")

In [ ]:
def calculer_duree(data, colonne_debut, colonne_fin):
    data[colonne_debut] = pd.to_datetime(data[colonne_debut], errors='coerce')
    data[colonne_fin] = pd.to_datetime(data[colonne_fin], errors='coerce')

    data['duree'] = data[colonne_fin] - data[colonne_debut]

    return data

def cyclical_encoding(X, max_value):
    return np.column_stack((
        np.sin(2 * np.pi * X / max_value),
        np.cos(2 * np.pi * X / max_value)
    ))

def season_encoding(month):
    if month in [12, 1, 2]:
        return 0  # Hiver
    elif month in [3, 4, 5]:
        return 1  # Printemps
    elif month in [6, 7, 8]:
        return 2  # Été
    else:
        return 3  # Automne

doubs_data = calculer_duree(doubs_data, 'date_debut', 'date_fin')
doubs_data['centroid'] = doubs_data.geometry.centroid
doubs_data['centroid_lon'] = doubs_data['centroid'].x
doubs_data['centroid_lat'] = doubs_data['centroid'].y
doubs_data['duree_minutes'] = pd.to_timedelta(doubs_data['duree']).dt.total_seconds() / 60

In [ ]:
print(caserne.crs)
print(compagnies.crs)

In [ ]:
caserne = caserne.to_crs(epsg=4326)
caserne["lon"] = caserne.geometry.x
caserne["lat"] = caserne.geometry.y

In [ ]:
compagnies = compagnies.to_crs(epsg=4326)
print(compagnies.geometry.head())

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
doubs_data.plot(ax=ax, color="blue", edgecolor="black", alpha=0.5, label="Polygones (Doubs)")
caserne.plot(ax=ax, color="red", markersize=50, edgecolor="black", label="Casernes")
ax.set_title("Carte des polygones dans Doubs avec Casernes")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
compagnies.plot(ax=ax, color="blue", edgecolor="black", alpha=0.5, label="Polygones (Doubs)")
caserne.plot(ax=ax, color="red", markersize=50, edgecolor="black", label="Casernes")
ax.set_title("Carte des polygones dans Doubs avec Casernes")
ax.legend()
plt.show()

In [ ]:
def get_nearest_caserne_distance(hexagon, caserne):
    distances = caserne.geometry.distance(hexagon)
    return distances.min()

doubs_data['distance_to_nearest_caserne'] = doubs_data.geometry.apply(get_nearest_caserne_distance, caserne=caserne)
print(doubs_data[['geometry', 'distance_to_nearest_caserne']].head())

In [ ]:
firepoint = pd.concat([firepoint_1, firepoint_2, firepoint_3], ignore_index=True)
firepoint = firepoint.drop(columns=['geometry'], errors='ignore')

doubs_data = doubs_data.merge(firepoint, on='hex_id', how='left')

print(doubs_data.head())

In [ ]:
doubs_data = doubs_data.drop(columns=['date', 'departement', 'geometry', 'duree', 'centroid', 'scale0', 'sinister', 'date_debut', 'date_fin', 'raison_sortie', 'label', 'year'], errors='ignore')

In [ ]:
# Calcul des statistiques pour chaque `hex_id`
doubs_moy = doubs_data.groupby('hex_id').agg(
    {'duree_minutes': ['mean', 'max', 'min', 'std'],
     'centroid_lon': 'mean',
     'centroid_lat': 'mean',
     'distance_to_nearest_caserne': 'mean',
     'NDVI': 'mean',
     'NDMI': 'mean',
     'NDBI': 'mean',
     'NDSI': 'mean',
     'NDWI': 'mean',
     'osmnx': 'mean',
     'PasDeRoute': 'mean',
     'motorway': 'mean',
     'primary': 'mean',
     'secondary': 'mean',
     'tertiary': 'mean',
     'path': 'mean',
     'population': 'mean',
     'elevation': 'mean',
     'foret_encoder': 'mean',
     'highway_encoder': 'mean',
     'foret': 'mean',
     'PasDeforet': 'mean',
     'Châtaignier': 'mean',
     'Chênes décidus': 'mean',
     'Conifères': 'mean',
     'Douglas': 'mean',
     'Feuillus': 'mean',
     'Hêtre': 'mean',
     'Mixte': 'mean',
     'Mélèze': 'mean',
     'NC': 'mean',
     'NR': 'mean',
     'Peuplier': 'mean',
     'Pin autre': 'mean',
     'Pin laricio, pin noir': 'mean',
     'Pin maritime': 'mean',
     'Pin sylvestre': 'mean',
     'Pins mélangés': 'mean',
     'Robinier': 'mean',
     'Sapin, épicéa': 'mean',
     'dynamic_world': 'mean',
     'water': 'mean',
     'tree': 'mean',
     'grass': 'mean',
     'crops': 'mean',
     'shrub': 'mean',
     'flooded': 'mean',
     'built': 'mean',
     'bare': 'mean',
     'snow': 'mean'}
).reset_index()

doubs_moy.columns = ['_'.join(col).strip() for col in doubs_moy.columns.values]
doubs_moy.head()

In [ ]:
print("Dimensions du DataFrame :")
print(doubs_moy.shape)

print("\nInfo sur les types de données et les valeurs manquantes :")
print(doubs_moy.info())

plt.figure(figsize=(10, 6))
sns.heatmap(doubs_moy.isnull(), cbar=False, cmap='viridis')
plt.title("Carte des valeurs manquantes (NaN)")
plt.show()

print("\nNombre de valeurs manquantes (NaN) par colonne :")
print(doubs_moy.isnull().sum())

In [ ]:
nan_percentage = doubs_moy.isnull().mean() * 100

cols_to_keep = nan_percentage[nan_percentage <= 80].index

doubs_moy = doubs_moy[cols_to_keep]

print("\nDimensions du DataFrame après suppression des colonnes avec plus de 80% de NaN :")
print(doubs_moy.shape)

print("\nColonnes restantes après nettoyage :")
print(doubs_moy.columns)

In [ ]:
doubs_moy = doubs_moy.dropna()

print("\nVérification des NaN après suppression des lignes dans doubs_moy :")
print(doubs_moy.isnull().sum())

print("\nDimensions du DataFrame doubs_moy après suppression des NaN :")
print(doubs_moy.shape)

In [ ]:
X = doubs_moy.drop(columns=["duree_minutes_mean", "hex_id_"])
y = doubs_moy["duree_minutes_mean"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Taille Train : {X_train.shape}, Taille Test : {X_test.shape}")

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"MAE: {mae:.2f} minutes")
print(f"RMSE: {rmse:.2f} minutes")

In [ ]:
corr_matrix = X_train.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, linewidths=0.5)
plt.title("Matrice de Corrélation des Features")
plt.show()

In [ ]:
seuil_corr = 0.9

corr_features = set()
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > seuil_corr:
            colname = corr_matrix.columns[i]
            corr_features.add(colname)

print("Features fortement corrélées à supprimer :", corr_features)

X_train_filtered = X_train.drop(columns=corr_features)
X_test_filtered = X_test.drop(columns=corr_features)

In [ ]:
model_filtered = RandomForestRegressor(n_estimators=100, random_state=42)
model_filtered.fit(X_train_filtered, y_train)

y_pred_filtered = model_filtered.predict(X_test_filtered)

mae_filtered = mean_absolute_error(y_test, y_pred_filtered)
rmse_filtered = mean_squared_error(y_test, y_pred_filtered, squared=False)

print(f"MAE après suppression des features corrélées: {mae_filtered:.2f} minutes")
print(f"RMSE après suppression des features corrélées: {rmse_filtered:.2f} minutes")

In [ ]:
xgb_model = XGBRegressor(objective="reg:squarederror", random_state=42)
xgb_model.fit(X_train_filtered, y_train)
y_pred_xgb = xgb_model.predict(X_test_filtered)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = mean_squared_error(y_test, y_pred_xgb, squared=False)

print(f"MAE avec XGBoost : {mae_xgb:.2f} minutes")
print(f"RMSE avec XGBoost : {rmse_xgb:.2f} minutes")

In [ ]:
# Création du modèle
xgb_regressor = XGBRegressor(
    objective='reg:squarederror',  # Fonction de perte adaptée à la régression
    n_estimators=100,  # Nombre d'arbres (peut être ajusté)
    learning_rate=0.1,  # Taux d’apprentissage
    max_depth=5,  # Profondeur max des arbres
    subsample=0.8,  # Pour éviter l’overfitting
    colsample_bytree=0.8,  # Sélection aléatoire de colonnes pour chaque arbre
    alpha=0.1,  # Régularisation L1
    lambda_=1.0  # Régularisation L2
)

# Entraînement
xgb_regressor.fit(X_train, y_train)


In [ ]:
# Prédiction sur le test set
y_pred = xgb_regressor.predict(X_test)

# Calcul des métriques
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# Définition des valeurs à tester
param_grid = {
    "n_estimators": [100, 300, 500],   # Nombre d'arbres
    "max_depth": [3, 5, 7],            # Profondeur max des arbres
    "learning_rate": [0.01, 0.1, 0.3], # Taux d'apprentissage
    "subsample": [0.7, 0.8, 1.0],      # Pourcentage des données utilisées par arbre
    "colsample_bytree": [0.7, 0.8, 1.0] # Pourcentage des features utilisées par arbre
}

# Création du modèle XGBoost
xgb_model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42)

# Recherche des meilleurs paramètres avec validation croisée
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=3,  # Validation croisée à 3 folds
    scoring="neg_mean_squared_error",  # Optimisation basée sur l'erreur quadratique moyenne
    verbose=1,
    n_jobs=-1  # Utilisation de tous les cœurs du CPU
)

# Entraînement de la recherche
grid_search.fit(X_train, y_train)

# Meilleurs paramètres trouvés
print("Meilleurs hyperparamètres :", grid_search.best_params_)